# Phase 2 Fixes Experiment Runner

**P0 实验**: 验证 M2 对齐和 robustness penalty 修正

- 16 losses × 3 seeds = 48 runs
- 预计运行时间: 24-36 小时
- 支持断点续跑

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
set -euo pipefail

# 2. Clone or update repo
if [ ! -d "/content/FYP" ]; then
  git clone https://github.com/ROUCHER27/FYP.git /content/FYP
fi

cd /content/FYP
git fetch origin
git checkout phase2-fixes
git pull origin phase2-fixes

echo "Branch: $(git branch --show-current)"
echo "Commit: $(git rev-parse --short HEAD)"

In [ ]:
%%bash
# 3. Install dependencies
cd /content/FYP
pip install -q -r requirements.txt

In [ ]:
%%bash
# 4. Verify Drive paths
DRIVE_ROOT="/content/drive/MyDrive/FYP/phase2-fixes"
mkdir -p "${DRIVE_ROOT}/results"
mkdir -p "${DRIVE_ROOT}/checkpoints"
mkdir -p "${DRIVE_ROOT}/logs"

echo "Drive root: ${DRIVE_ROOT}"
ls -la "${DRIVE_ROOT}"

In [ ]:
%%bash
# 5. Smoke test (1 loss, 2 epochs, 2 months)
cd /content/FYP

python run_phase2_robustness.py \
  --losses imadl_m2_alpha02 \
  --seeds 42 \
  --test-months 2 \
  --max-epochs 2 \
  --batch-size 1024 \
  --max-weight 0.05 \
  --matrix-mode light \
  --output-dir /content/drive/MyDrive/FYP/phase2-fixes/results \
  --checkpoint-dir /content/drive/MyDrive/FYP/phase2-fixes/checkpoints \
  --skip-existing \
  --resume-mode auto

echo "✅ Smoke test passed"

In [ ]:
%%bash
set -euo pipefail

# 6. Full Phase 2.1 experiment (48 runs)
cd /content/FYP

LOSSES="imadl_m2_alpha02,imadl_m2_alpha03,imadl_m2_alpha05,imadl_m2_alpha07,"
LOSSES+="imadl_gmadl_beta02,imadl_gmadl_beta03,imadl_gmadl_beta05,imadl_gmadl_beta07,"
LOSSES+="m2_robust_gamma001,m2_robust_gamma01,m2_robust_gamma05,m2_robust_gamma10,"
LOSSES+="adaptive_lambda001,adaptive_lambda01,adaptive_lambda05,adaptive_lambda100"

python run_phase2_robustness.py \
  --losses "${LOSSES}" \
  --seeds 42,52,62 \
  --test-months 24 \
  --max-epochs 20 \
  --batch-size 1024 \
  --max-weight 0.05 \
  --matrix-mode light \
  --output-dir /content/drive/MyDrive/FYP/phase2-fixes/results \
  --checkpoint-dir /content/drive/MyDrive/FYP/phase2-fixes/checkpoints \
  --skip-existing \
  --resume-mode auto \
  2>&1 | tee /content/drive/MyDrive/FYP/phase2-fixes/logs/phase21_$(date +%Y%m%d_%H%M%S).log

In [ ]:
%%bash
# 7. Check results
RESULTS_DIR="/content/drive/MyDrive/FYP/phase2-fixes/results"

echo "=== Completed runs ==="
find "${RESULTS_DIR}" -name "sanity_summary_*.json" | wc -l

echo "\n=== Sample results ==="
find "${RESULTS_DIR}" -name "sanity_summary_*.json" | head -3 | xargs -I {} sh -c 'echo "File: {}"; cat {}; echo'